# 3단계 + 추가 HP: 인코딩 구조 & 하이퍼파라미터 검증

**전제**: 1단계(RoBERTa 선정) + 2단계(LR/LS/Pooling 결과) 확정 후 실행

| 구분 | 실험 | 변경 변수 | 질문 | 타겟 4축 |
|---|---|---|---|---|
| 추가HP | M2-H1 | Dropout 0.1 | 0.3이 최적인가? | — |
| 추가HP | M2-H2 | Dropout 0.5 | 더 강한 정규화가 필요한가? | — |
| 추가HP | M2-H3 | Head LR 3e-5 | 헤드를 더 빠르게 학습시키면? | — |
| 3단계 | M3-B | Ending Pooling | 마무리 부분 집중이 효과 있나? | 에스칼레이션, 마무리 톤 |
| 3단계 | M3-C | Speaker-aware | 화자 구분이 비대칭 감지에 도움되나? | 발화 대칭성, 존댓말 비대칭 |
| 3단계 | M3-D | Multi-head | 대등성+마무리+위협의도 별도 판단 → 결합 | 종합 (4축 전체) |

In [ ]:
# ============================================================
# Cell 1: 환경 설정
# ============================================================
import pandas as pd
import numpy as np
import re
import os
import time
import gc
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Colab 한글 폰트
import subprocess
subprocess.run(['apt-get', '-y', 'install', 'fonts-nanum'], capture_output=True)
import matplotlib.font_manager as fm
for font in fm.findSystemFonts(fontpaths=['/usr/share/fonts']):
    fm.fontManager.addfont(font)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ============================================================
# Cell 2: 데이터 로드
# ============================================================
if not os.path.exists('/content/DLThon'):
    !git clone https://github.com/seongyeon-h/DLThon.git /content/DLThon

DATA_DIR = '/content/DLThon/data'
train_df = pd.read_csv(f'{DATA_DIR}/train_final.csv')
val_df = pd.read_csv(f'{DATA_DIR}/val_final.csv')
test_df_raw = pd.read_csv(f'{DATA_DIR}/test.csv')

label2id = {'갈취 대화':0, '기타 괴롭힘 대화':1, '직장 내 괴롭힘 대화':2, '협박 대화':3, '일반 대화':4}
id2label = {v:k for k,v in label2id.items()}

def preprocess(text):
    if not isinstance(text, str): return ''
    return re.sub(r'\s+', ' ', text.replace('\n', ' ')).strip()

train_df['text'] = train_df['conversation'].apply(preprocess)
train_df['label'] = train_df['class'].map(label2id)
val_df['text'] = val_df['conversation'].apply(preprocess)
val_df['label'] = val_df['class'].map(label2id)
test_df_raw['text'] = test_df_raw['conversation'].apply(lambda x: preprocess(str(x).replace('"','')))

train_texts, train_labels = train_df['text'].tolist(), train_df['label'].tolist()
val_texts, val_labels = val_df['text'].tolist(), val_df['label'].tolist()
test_texts = test_df_raw['text'].tolist()

print(f'Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}')

In [ ]:
# ============================================================
# Cell 3: 고정값 + 실험 목록
# ============================================================
MODEL_NAME = 'klue/roberta-base'
MAX_LEN = 256
BATCH_SIZE = 32
EPOCHS = 3
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

# 2단계 확정값 (결과 나오면 여기 업데이트)
BEST_BACKBONE_LR = 4e-6
BEST_HEAD_LR = 2e-5
BEST_LS = 0.0
BEST_DROPOUT = 0.3
BEST_POOLING = 'mean'

# (실험ID, 설명, backbone_lr, head_lr, ls, dropout, pooling)
EXPERIMENTS = [
    # --- 추가 HP: Dropout ---
    ('M2-H1', 'Dropout 0.1',     BEST_BACKBONE_LR, BEST_HEAD_LR, BEST_LS, 0.1,  BEST_POOLING),
    ('M2-H2', 'Dropout 0.5',     BEST_BACKBONE_LR, BEST_HEAD_LR, BEST_LS, 0.5,  BEST_POOLING),
    # --- 추가 HP: Head LR ---
    ('M2-H3', 'Head LR 3e-5',    BEST_BACKBONE_LR, 3e-5,         BEST_LS, BEST_DROPOUT, BEST_POOLING),
    # --- 3단계: 인코딩 구조 ---
    ('M3-B',  'Ending Pooling',   BEST_BACKBONE_LR, BEST_HEAD_LR, BEST_LS, BEST_DROPOUT, 'ending'),
    ('M3-C',  'Speaker-aware',    BEST_BACKBONE_LR, BEST_HEAD_LR, BEST_LS, BEST_DROPOUT, 'speaker'),
    ('M3-D',  'Multi-head',       BEST_BACKBONE_LR, BEST_HEAD_LR, BEST_LS, BEST_DROPOUT, 'multihead'),
]

print(f'모델: {MODEL_NAME} (고정)')
print(f'기준: backbone_lr={BEST_BACKBONE_LR}, head_lr={BEST_HEAD_LR}, LS={BEST_LS}, dropout={BEST_DROPOUT}, pooling={BEST_POOLING}')
print(f'\n실험 {len(EXPERIMENTS)}종:')
for eid, desc, blr, hlr, ls, dr, pool in EXPERIMENTS:
    print(f'  {eid}: {desc}')

In [ ]:
# ============================================================
# Cell 4: Dataset + 모델 (Mean/CLS/Ending/Speaker/Multi-head) + 함수
# ============================================================

class ConversationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label': torch.tensor(self.labels[idx], dtype=torch.long)}


class MultiHeadClassifier(nn.Module):
    """
    M3-D: Multi-head 분류기
    
    3개의 전문 헤드가 각각 다른 관점에서 판단하고, 최종 결합하여 분류.
    
    [구조]
    RoBERTa → Mean Pooling (768차원)
                │
                ├── Head 1 (대등성): FC(768→128→1) → "대화 관계가 대등한가?" 점수
                ├── Head 2 (마무리톤): FC(768→128→1) → "마무리가 평화로운가?" 점수
                └── Head 3 (위협의도): FC(768→128→1) → "위협 의도가 있는가?" 점수
                │
                └── 결합: concat([pooled, h1, h2, h3]) → FC(771→256→5) → 5클래스 분류
    
    [설계 의도]
    - 단일 헤드는 모든 신호를 한번에 처리 → 중요 신호가 묻힐 수 있음
    - 3개 헤드가 각각 대등성/마무리/위협에 집중 → 경계선 대화 구분 개선
    - 최종 분류기가 3개 점수 + 원본 표현을 종합하여 판단
    
    [한계]
    - 3개 헤드에 별도 레이블이 없어 5클래스 loss 하나로 학습
    - 각 헤드가 의도대로 역할 분담할지는 학습 결과로 확인해야 함
    """
    def __init__(self, model_name, num_classes=5, dropout_rate=0.3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        h = self.backbone.config.hidden_size  # 768
        
        # --- 관점별 전문 헤드 (각각 하나의 점수 출력) ---
        
        # Head 1: 대등성 판단 (화법 대칭, 굴복/대등 비율)
        # "양쪽이 비슷한 톤으로 대화하고 있는가?"
        self.equality_head = nn.Sequential(
            nn.Linear(h, 128),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1),    # 1차원 점수 출력
        )
        
        # Head 2: 마무리 톤 판단 (에스칼레이션, 감정 곡선)
        # "대화가 평화롭게 끝나고 있는가?"
        self.ending_head = nn.Sequential(
            nn.Linear(h, 128),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1),
        )
        
        # Head 3: 위협 의도 판단 (위협 키워드, 공격성)
        # "실제 위협/공격 의도가 있는가?"
        self.threat_head = nn.Sequential(
            nn.Linear(h, 128),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1),
        )
        
        # --- 최종 결합 분류기 ---
        # 원본 표현(768) + 3개 헤드 점수(3) = 771차원 입력
        self.layer_norm = nn.LayerNorm(h)
        self.final_classifier = nn.Sequential(
            nn.Linear(h + 3, 256),    # 768 + 3 = 771
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes),
        )
    
    def mean_pooling(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        return (last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    
    def forward(self, input_ids, attention_mask):
        # Step 1: RoBERTa 인코딩
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.mean_pooling(out.last_hidden_state, attention_mask)  # (B, 768)
        pooled = self.layer_norm(pooled)
        
        # Step 2: 3개 전문 헤드 각각 점수 계산
        eq_score = self.equality_head(pooled)     # (B, 1) 대등성 점수
        end_score = self.ending_head(pooled)      # (B, 1) 마무리 톤 점수
        threat_score = self.threat_head(pooled)    # (B, 1) 위협 의도 점수
        
        # Step 3: 원본 표현 + 3개 점수 결합
        combined = torch.cat([pooled, eq_score, end_score, threat_score], dim=-1)  # (B, 771)
        
        # Step 4: 최종 5클래스 분류
        logits = self.final_classifier(combined)   # (B, 5)
        
        return logits


class ConversationClassifier(nn.Module):
    """
    백본 모델 + 풀링 전략 선택 (Mean/CLS/Ending/Speaker)
    Multi-head는 별도 클래스(MultiHeadClassifier)로 분리
    """
    def __init__(self, model_name, num_classes=5, dropout_rate=0.3, pooling='mean'):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.pooling = pooling
        h = self.backbone.config.hidden_size  # 768
        
        # Speaker-aware: 두 화자의 표현을 concat → 2*h
        input_dim = h * 2 if pooling == 'speaker' else h
        
        self.layer_norm = nn.LayerNorm(input_dim)
        self.fc1 = nn.Linear(input_dim, 256)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.last_hidden_state  # (B, seq_len, 768)
        mask = attention_mask.unsqueeze(-1).float()
        
        if self.pooling == 'mean':
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        
        elif self.pooling == 'cls':
            pooled = hidden[:, 0, :]
        
        elif self.pooling == 'ending':
            # 마지막 64토큰 (~3턴)만 풀링 → 대화 마무리에 집중
            seq_lens = attention_mask.sum(dim=1)
            ending_pooled = []
            for i in range(hidden.size(0)):
                end = int(seq_lens[i].item())
                start = max(0, end - 64)
                chunk = hidden[i, start:end, :]
                ending_pooled.append(chunk.mean(dim=0))
            pooled = torch.stack(ending_pooled)
        
        elif self.pooling == 'speaker':
            # 홀수 위치(화자A) vs 짝수 위치(화자B) 분리 풀링
            B_size, seq_len, h_dim = hidden.shape
            positions = torch.arange(seq_len, device=hidden.device).unsqueeze(0).expand(B_size, -1)
            odd_mask = ((positions % 2 == 0) & (attention_mask == 1)).unsqueeze(-1).float()
            even_mask = ((positions % 2 == 1) & (attention_mask == 1)).unsqueeze(-1).float()
            a_pooled = (hidden * odd_mask).sum(1) / odd_mask.sum(1).clamp(min=1e-9)
            b_pooled = (hidden * even_mask).sum(1) / even_mask.sum(1).clamp(min=1e-9)
            pooled = torch.cat([a_pooled, b_pooled], dim=-1)
        
        pooled = self.layer_norm(pooled)
        return self.fc2(self.dropout(self.activation(self.fc1(pooled))))


def create_model(model_name, num_classes, dropout_rate, pooling):
    """풀링 타입에 따라 적절한 모델 클래스를 선택하여 생성"""
    if pooling == 'multihead':
        return MultiHeadClassifier(model_name, num_classes, dropout_rate)
    else:
        return ConversationClassifier(model_name, num_classes, dropout_rate, pooling=pooling)


def train_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss, preds, trues = 0, [], []
    for batch in loader:
        ids, mask, labels = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['label'].to(device)
        optimizer.zero_grad()
        logits = model(ids, mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()
        preds.extend(logits.argmax(-1).cpu().numpy())
        trues.extend(labels.cpu().numpy())
    return total_loss / len(loader), f1_score(trues, preds, average='macro')


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds, trues = 0, [], []
    with torch.no_grad():
        for batch in loader:
            ids, mask, labels = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['label'].to(device)
            logits = model(ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            preds.extend(logits.argmax(-1).cpu().numpy())
            trues.extend(labels.cpu().numpy())
    return total_loss / len(loader), f1_score(trues, preds, average='macro'), preds, trues


print('Dataset, Model(5종 풀링 + Multi-head), 학습/검증 함수 정의 완료')

In [ ]:
# ============================================================
# Cell 5: 6종 자동 순회 학습
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_dataset = ConversationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = ConversationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = ConversationDataset(test_texts, [0]*len(test_texts), tokenizer, MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

all_results = []

for exp_id, desc, backbone_lr, head_lr, ls, dropout, pooling in EXPERIMENTS:
    print(f'\n{"="*70}')
    print(f'[{exp_id}] {desc}')
    print(f'  blr={backbone_lr}, hlr={head_lr}, LS={ls}, dropout={dropout}, pooling={pooling}')
    print(f'{"="*70}')
    
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    
    # create_model로 풀링/Multi-head 분기
    model = create_model(MODEL_NAME, len(label2id), dropout, pooling).to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f'  파라미터: {total_params:,}개')
    
    # 차등 학습률 — Multi-head도 동일 규칙 적용
    # (backbone = 보수적, 나머지 = 적극적)
    head_p, backbone_p = [], []
    for name, param in model.named_parameters():
        if 'backbone' in name:
            backbone_p.append(param)
        else:
            head_p.append(param)
    
    optimizer = torch.optim.AdamW([
        {'params': backbone_p, 'lr': backbone_lr, 'weight_decay': WEIGHT_DECAY},
        {'params': head_p, 'lr': head_lr, 'weight_decay': WEIGHT_DECAY},
    ])
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
    criterion = nn.CrossEntropyLoss(label_smoothing=ls)
    
    best_f1, best_epoch = 0, 0
    history = []
    save_path = f'best_model_{exp_id}.pt'
    total_start = time.time()
    
    for epoch in range(EPOCHS):
        start = time.time()
        train_loss, train_f1 = train_epoch(model, train_loader, criterion, optimizer, scheduler, device)
        val_loss, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
        elapsed = time.time() - start
        history.append({'epoch': epoch+1, 'train_loss': train_loss, 'train_f1': train_f1,
                        'val_loss': val_loss, 'val_f1': val_f1})
        marker = ''
        if val_f1 > best_f1:
            best_f1 = val_f1; best_epoch = epoch + 1
            torch.save(model.state_dict(), save_path)
            marker = ' << BEST'
        print(f'  Epoch {epoch+1} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | {elapsed:.0f}s{marker}')
    
    total_time = (time.time() - total_start) / 60
    
    # Best Model 평가
    model.load_state_dict(torch.load(save_path, weights_only=True))
    _, final_f1, final_preds, final_trues = evaluate(model, val_loader, criterion, device)
    class_names = [id2label[i] for i in range(5)]
    report = classification_report(final_trues, final_preds, target_names=class_names, digits=4, output_dict=True)
    normal_f1 = report['일반 대화']['f1-score']
    
    print(f'\n  >> Best: Epoch {best_epoch} | Val F1: {best_f1:.4f} | 일반대화 F1: {normal_f1:.4f} | {total_time:.1f}분')
    print(classification_report(final_trues, final_preds, target_names=class_names, digits=4))
    
    # Test 추론
    model.eval()
    test_preds = []
    with torch.no_grad():
        for batch in test_loader:
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            test_preds.extend(logits.argmax(-1).cpu().numpy())
    test_classes = [str(id2label[p]) for p in test_preds]
    test_normal = sum(1 for c in test_classes if c == '일반 대화')
    print(f'  >> Test 일반대화: {test_normal}건 / 100건')
    print(f'  >> Test 분포: {pd.Series(test_classes).value_counts().to_dict()}')
    
    all_results.append({
        'exp_id': exp_id, 'desc': desc, 'pooling': pooling, 'dropout': dropout,
        'backbone_lr': backbone_lr, 'head_lr': head_lr, 'ls': ls,
        'val_f1': round(best_f1, 4), 'normal_val_f1': round(normal_f1, 4),
        'test_normal': test_normal, 'best_epoch': best_epoch,
        'time_min': round(total_time, 1), 'history': history,
    })
    
    del model, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'\n{"="*70}')
print('3단계 + 추가 HP 완료!')
print(f'{"="*70}')

In [ ]:
# ============================================================
# Cell 6: 결과 비교
# ============================================================

# M1-A (기준) 결과 추가
baseline_ref = {'exp_id': 'M1-A', 'desc': '기준 (Mean Pooling)', 'pooling': BEST_POOLING,
                'dropout': BEST_DROPOUT, 'backbone_lr': BEST_BACKBONE_LR, 'head_lr': BEST_HEAD_LR,
                'ls': BEST_LS, 'val_f1': 0.9299, 'normal_val_f1': 0.9949,
                'test_normal': 74, 'best_epoch': 3, 'time_min': 35.1}

all_with_base = [baseline_ref] + [{k:v for k,v in r.items() if k != 'history'} for r in all_results]
comp = pd.DataFrame(all_with_base).sort_values('val_f1', ascending=False)

print('\n' + '=' * 70)
print('3단계 + 추가 HP 결과 (Val F1 내림차순)')
print('=' * 70)
print(comp[['exp_id','desc','dropout','pooling','val_f1','test_normal','best_epoch']].to_string(index=False))

# Dropout 비교
print('\n[Dropout 비교] 0.1 vs 0.3(기준) vs 0.5')
for eid in ['M2-H1', 'M1-A', 'M2-H2']:
    r = comp[comp['exp_id']==eid]
    if len(r) > 0:
        r = r.iloc[0]
        print(f'  {r["exp_id"]} dropout={r["dropout"]}: Val F1={r["val_f1"]:.4f}, Test 일반={r["test_normal"]}건')

# Head LR 비교
print('\n[Head LR 비교] 2e-5(기준) vs 3e-5')
for eid in ['M1-A', 'M2-H3']:
    r = comp[comp['exp_id']==eid]
    if len(r) > 0:
        r = r.iloc[0]
        print(f'  {r["exp_id"]} head_lr={r["head_lr"]}: Val F1={r["val_f1"]:.4f}, Test 일반={r["test_normal"]}건')

# 풀링/구조 비교 (M3-D 포함)
print('\n[인코딩 구조 비교] Mean(기준) vs Ending vs Speaker-aware vs Multi-head')
for eid in ['M1-A', 'M3-B', 'M3-C', 'M3-D']:
    r = comp[comp['exp_id']==eid]
    if len(r) > 0:
        r = r.iloc[0]
        print(f'  {r["exp_id"]} {r["pooling"]}: Val F1={r["val_f1"]:.4f}, Test 일반={r["test_normal"]}건')

comp.to_csv('stage3_hp_results.csv', index=False)
print('\n결과 저장: stage3_hp_results.csv')

In [ ]:
# ============================================================
# Cell 7: 시각화
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Dropout
dr_data = comp[comp['exp_id'].isin(['M2-H1','M1-A','M2-H2'])].sort_values('dropout')
colors_dr = ['#94a3b8' if e != 'M1-A' else '#3b82f6' for e in dr_data['exp_id']]
axes[0].bar([f'0.1\n(M2-H1)', f'0.3\n(기준)', f'0.5\n(M2-H2)'], dr_data['val_f1'].values, color=colors_dr)
axes[0].set_title('Dropout 비교', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Val F1')
if len(dr_data) > 0:
    axes[0].set_ylim(dr_data['val_f1'].min()-0.01, dr_data['val_f1'].max()+0.005)
    for i, v in enumerate(dr_data['val_f1']): axes[0].text(i, v+0.001, f'{v:.4f}', ha='center')

# Head LR
hlr_data = comp[comp['exp_id'].isin(['M1-A','M2-H3'])]
colors_hlr = ['#3b82f6', '#94a3b8']
axes[1].bar(['2e-5\n(기준)', '3e-5\n(M2-H3)'], hlr_data['val_f1'].values, color=colors_hlr)
axes[1].set_title('Head LR 비교', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Val F1')
if len(hlr_data) > 0:
    axes[1].set_ylim(hlr_data['val_f1'].min()-0.01, hlr_data['val_f1'].max()+0.005)
    for i, v in enumerate(hlr_data['val_f1']): axes[1].text(i, v+0.001, f'{v:.4f}', ha='center')

# 풀링/구조 (4종: Mean, Ending, Speaker, Multi-head)
pool_ids = ['M1-A', 'M3-B', 'M3-C', 'M3-D']
pool_data = comp[comp['exp_id'].isin(pool_ids)]
colors_pool = ['#3b82f6', '#10b981', '#f59e0b', '#ef4444']
labels_pool = [f'{r["pooling"]}\n({r["exp_id"]})' for _, r in pool_data.iterrows()]
axes[2].bar(labels_pool, pool_data['val_f1'].values, color=colors_pool[:len(pool_data)])
axes[2].set_title('인코딩 구조 비교', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Val F1')
if len(pool_data) > 0:
    axes[2].set_ylim(pool_data['val_f1'].min()-0.01, pool_data['val_f1'].max()+0.005)
    for i, v in enumerate(pool_data['val_f1']): axes[2].text(i, v+0.001, f'{v:.4f}', ha='center')

plt.suptitle('3단계 + 추가 HP Ablation 결과', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('stage3_hp_charts.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Cell 8: Test 일반대화 비교
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))
sorted_comp = comp.sort_values('test_normal', ascending=True)
colors = ['#3b82f6' if eid == 'M1-A' else '#94a3b8' for eid in sorted_comp['exp_id']]
ax.barh(sorted_comp['exp_id'] + ' ' + sorted_comp['desc'], sorted_comp['test_normal'], color=colors)
ax.axvline(x=100, color='red', linestyle='--', alpha=0.5, label='목표 (100건)')
ax.axvline(x=74, color='blue', linestyle='--', alpha=0.5, label='기준 M1-A (74건)')
ax.set_xlabel('Test 일반대화 예측 수')
ax.set_title('Test 일반대화 탐지 — 기준(74건) 대비', fontsize=13, fontweight='bold')
ax.legend()
for i, v in enumerate(sorted_comp['test_normal']):
    ax.text(v+1, i, f'{v}건', va='center', fontsize=11)
plt.tight_layout()
plt.savefig('stage3_test_normal.png', dpi=150, bbox_inches='tight')
plt.show()